In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import countDistinct

# Sample dataset
data = [
    Row(category="A", subcategory="X", value=10),
    Row(category="A", subcategory="Y", value=20),
    Row(category="A", subcategory="X", value=30),
    Row(category="B", subcategory="X", value=40),
    Row(category="B", subcategory="Y", value=50),
    Row(category="B", subcategory="Y", value=60),
    Row(category="C", subcategory="Z", value=70),
    Row(category="C", subcategory="Z", value=80)
]
df = spark.createDataFrame(data)
display(df)


In [0]:
from pyspark.sql.functions import count
# groupBy().count()
group_count1 = df.groupBy("category").count()
group_count2 = df.groupBy("category").agg(count("category"))
display(group_count2)
# select category,count(*) from table group by category

In [0]:

# countDistinct()
distinct_count = df.groupBy("category").agg(countDistinct("subcategory").alias("distinct_subcategories"))
display(distinct_count)
# select count(distinct subcategory) from table


In [0]:

# pivot()
pivot_df = df.groupBy("category").pivot("subcategory").sum("value")
#pivot_df = df.groupBy("category").pivot("subcategory").count()
display(pivot_df)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum,count,countDistinct,avg
spark = SparkSession.builder.appName("Demo").getOrCreate()
data = [
    ("North", "Apple", 100),
    ("North", "Banana", 150),
    ("South", "Apple", 200),
    ("South", "Banana", 50),
    ("East", "Apple", 120),
    ("East", "Banana", 80),
    ("West", "Apple", 90),
    ("West", "Banana", 110),
    ("North", "Apple", 130),
    ("South", "Banana", 70)
]
df = spark.createDataFrame(data,["Region","Product","Price"])
display(df)

In [0]:
df.groupBy("Region").agg(avg("Price").alias("avg")).show()

In [0]:
#df1 = df.groupBy("Region").count()
df1 = df.groupBy("Product").agg(count("Product").alias("product"))
df1.show()

In [0]:
df.groupBy("Region").agg(countDistinct("product").alias("CD")).show()

In [0]:
df.groupBy("Region").pivot("product").agg(sum("price")).show()

In [0]:
df = spark.read.option("header",True).csv("/Volumes/sampleproject/default/ramvolume/sales_data/transactions_raw.csv")
display(df)

In [0]:
from pyspark.sql.functions import when, col

df = df.filter(~col("amount").rlike("[A-Za-z]")).withColumn("amount", col("amount").cast("int"))
display(df)

In [0]:
# Example: Use expr('try_cast') in filter for Spark DataFrame
from pyspark.sql.functions import expr

# Keep only rows where amount can be safely cast to BIGINT
filtered_df = df.filter(expr("try_cast(amount AS BIGINT) IS NOT NULL"))
filtered_df.show()

# Or, keep rows where amount can be DOUBLE
filtered_double_df = df.filter(expr("try_cast(amount AS DOUBLE) IS NOT NULL"))
filtered_double_df.show()

# You can also use withColumn to get a safely casted column:
df_with_cast = df.withColumn("amount_bigint", expr("try_cast(amount AS BIGINT)"))
df_with_cast.show()


In [0]:
from pyspark.sql.functions import avg, col
# Filter rows where amount contains only valid numeric (including decimals)
#df_valid = df.filter(col("amount").rlike("^[0-9]+(\\.[0-9]+)?$"))
df1 = df.groupBy("customer_id").agg(avg(col("amount").cast("double")).alias("avg_amount"))
display(df1)